# Aarohan-350M — Kaggle Evaluation Notebook

**Run AFTER SFT is complete.**

Evaluates the final SFT model on:
- **Qualitative tests** — multi-language SE questions (Python, JS, SQL, Java)
- **HumanEval** — 164 Python coding problems (industry-standard benchmark)
- **Perplexity** — validation loss on held-out code data

**Instructions:**
1. Enable GPU: Settings → Accelerator → **T4 GPU** (no TPU needed)
2. Attach the SFT notebook output (which contains `sft_final.pt`) as input
3. Add `se-llm-data` dataset (for tokenizer.json)
4. Click **Run All**

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
!pip install -q datasets tokenizers pyyaml

In [ ]:
# ── Cell 2: Clone Aarohan-350M code + link tokenizer ──────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token = secrets.get_secret('GITHUB_TOKEN')
        url   = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {url} /kaggle/working/se-llm-350m
else:
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m

# Link tokenizer
os.makedirs('tokenizer', exist_ok=True)
tok_candidates = [
    '/kaggle/input/datasets/vedase/se-llm-data/tokenizer.json',
    '/kaggle/input/se-llm-data/tokenizer.json',
]
for tok_src in tok_candidates:
    if os.path.exists(tok_src):
        tok_dst = 'tokenizer/tokenizer.json'
        if not os.path.exists(tok_dst):
            os.symlink(tok_src, tok_dst)
        print(f'✅ Tokenizer linked from {tok_src}')
        break
else:
    print('❌ tokenizer.json not found — add the se-llm-data dataset!')

In [ ]:
# ── Cell 3: Load Aarohan-350M SFT checkpoint ──────────────────
# Attach the OUTPUT of your SFT notebook as input in the sidebar.
import os, shutil, glob, torch, sys

sys.path.insert(0, '/kaggle/working/se-llm-350m')
os.makedirs('checkpoints_sft', exist_ok=True)

# Auto-search for sft_final.pt or latest.pt in all input paths
search = (
    glob.glob('/kaggle/input/**/sft_final.pt', recursive=True) +
    glob.glob('/kaggle/input/**/latest.pt',    recursive=True)
)

sft_ckpt = None
if search:
    src = search[0]
    dst = f'checkpoints_sft/{os.path.basename(src)}'
    if not os.path.exists(dst):
        print(f'Copying {os.path.basename(src)}... (2.36 GB, please wait)')
        shutil.copy2(src, dst)
    sft_ckpt = dst
    print(f'✅ Found SFT checkpoint: {src}')
else:
    print('❌ No SFT checkpoint found!')
    print('   → Attach your SFT notebook output in the sidebar (+ Add Data → Notebooks)')

assert sft_ckpt, 'No SFT checkpoint found!'

from evaluation.generate import load_model_from_checkpoint, load_tokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model_from_checkpoint(sft_ckpt, device)
tokenizer  = load_tokenizer('tokenizer/tokenizer.json')
print(f'\n✅ Aarohan-350M ready on {device}')

In [ ]:
# ── Cell 4: Qualitative Tests ─────────────────────────────────
# Test Aarohan across different SE domains
from evaluation.generate import chat_turn

test_cases = [
    ('Python',      'Write a Python function to check if a string is a palindrome.'),
    ('JavaScript',  'Write a JavaScript function to debounce a function call.'),
    ('SQL',         'Write a SQL query to find all users who placed more than 3 orders in the last 30 days.'),
    ('Java',        'Write a Java method to check if a binary tree is balanced.'),
    ('Code Review', 'Find any bugs in this Python code: def divide(a, b): return a/b'),
    ('CS Concept',  'What is the difference between a process and a thread?'),
]

print('=== QUALITATIVE EVALUATION ===\n')
for lang, prompt in test_cases:
    print(f'[{lang}] {prompt}')
    response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=300)
    print(f'Aarohan:\n{response}')
    print('─' * 60)

In [ ]:
# ── Cell 5: HumanEval Benchmark ───────────────────────────────
# 164 Python coding problems — industry-standard code benchmark.
# Expected score for a well-trained 350M model: 10-20% pass@1.
from evaluation.humaneval import run_humaneval

print('=== HUMANEVAL BENCHMARK ===')
print('Running 164 Python coding problems...\n')

results = run_humaneval(
    checkpoint=sft_ckpt,
    tokenizer_path='tokenizer/tokenizer.json',
    temperature=0.2,        # low temperature for evaluation accuracy
    output_file='evaluation/humaneval_results.jsonl',
)

print(f'\nHumanEval pass@1: {results.get("pass@1", 0)*100:.1f}%')

In [ ]:
# ── Cell 6: Perplexity on validation set ──────────────────────
import math
from training.dataset import build_dataloader, estimate_loss

# Look for val.bin in Kaggle dataset
val_candidates = [
    '/kaggle/input/datasets/vedase/se-llm-data/val.bin',
    '/kaggle/input/se-llm-data/val.bin',
]
val_bin = next((p for p in val_candidates if os.path.exists(p)), None)

if val_bin:
    os.makedirs('data/processed', exist_ok=True)
    val_link = 'data/processed/val.bin'
    if not os.path.exists(val_link):
        os.symlink(val_bin, val_link)

    val_loader = build_dataloader(val_link, 2048, batch_size=4, shuffle=False)
    losses     = estimate_loss(model, val_loader, val_loader, eval_batches=50, device=device)
    perplexity = math.exp(losses['val'])

    print(f'Validation loss: {losses["val"]:.4f}')
    print(f'Perplexity:      {perplexity:.2f}')
else:
    losses     = {'val': float('nan')}
    perplexity = float('nan')
    print('val.bin not found — skipping perplexity')

In [ ]:
# ── Cell 7: Final Benchmark Summary ───────────────────────────
print('\n' + '='*55)
print('  Aarohan-350M — Final Benchmark Results')
print('='*55)
print(f'  HumanEval pass@1:  {results.get("pass@1", 0)*100:.1f}%')
print(f'  Validation loss:   {losses["val"]:.4f}')
print(f'  Perplexity:        {perplexity:.2f}')
print('='*55)
print()
print('Compare with other models:')
print('  GPT-2 (124M):     pass@1 ~0%   (base model, no SFT)')
print('  CodeBERT (125M):  not applicable (encoder-only)')
print('  CodeGen-350M:     pass@1 ~8-12%')
print('  Aarohan-350M:     ???')